# Étape D : la table finale (un logement par ligne) et sa représentation TF-IDF

**Objectif :** construire la table utilisée par les modèles : **une ligne par logement**, avec ses avis
prétraités mis bout à bout et ses notes. Puis décrire sa représentation TF-IDF.

On part de `reviews_ablation.parquet` (étape C, une ligne par avis). La fonction de regroupement est dans `src/agregation.py`.

Plan :
1. Les avis prétraités
2. Pourquoi regrouper les avis par logement
3. La table finale
4. Représentation TF-IDF
5. Sauvegarde

In [1]:
import sys
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.append('../src')
import agregation as ag

pd.set_option('display.max_colwidth', 150)

df = pd.read_parquet('../data/processed/reviews_ablation.parquet')
print(len(df), 'avis,', df['listing_id'].nunique(), 'logements')

337787 avis, 9862 logements


## 1. Les avis prétraités

Une ligne par avis : le logement, les versions du texte (étape C) et les notes du logement.

In [2]:
df[['listing_id', 'comments_clean', 'text_neg', 'review_scores_cleanliness']].head()

,listing_id,comments_clean,text_neg,review_scores_cleanliness
0,222887,cet appartement est tres bien situé dans le centre de bordeaux. nous avions loué pour 4nuits mais les 2premieres nuits ont été très difficiles car...,tres bien situer centre louer mais nuit très difficile bruit sourd type essorage lav l immeuble cause ventilation installer dernier étage couchage...,4.76
1,222887,"merci beaucoup pour l'accueil. l'appartement est superbe, mes amies et moi avons beaucoup aimé la décoration, la vue est magnifique, le logement e...",beaucoup accueil superbe amie beaucoup aimer décoration vue magnifique très bien situer beaucoup,4.76
2,222887,"très bel appartement bien situé sur bordeaux en face de la porte de bourgogne. grande pièce a vivre et belle petite terrasse, c'est parfait pour s...",très bel bien situer face porte bourgogne grand pièce vivre bel petit terrasse parfait reposer long promenade pied vélo gironde marché capucin moi...,4.76
3,222887,belle découverte et suzanna est très réactive.,bel découverte HOTE très réactif,4.76
4,222887,nous avons passé un très bon séjour ! l'appartement a beaucoup de charme et est très bien équipé. la terrasse est exceptionnelle. l'appartement es...,passer très bon beaucoup charme très bien équiper terrasse exceptionnel très bien situer très proche tram commerce restaurant proximité attention ...,4.76


## 2. Pourquoi regrouper les avis par logement

Au TD5, chaque tweet avait sa propre étiquette. Ici, les notes (propreté, emplacement...) sont données
au **logement**, pas à l'avis : tous les avis d'un logement portent la même note.

In [3]:
exemple = df['listing_id'].iloc[0]
df.loc[df['listing_id'] == exemple, ['listing_id', 'text_neg', 'review_scores_cleanliness']].head()

,listing_id,text_neg,review_scores_cleanliness
0,222887,tres bien situer centre louer mais nuit très difficile bruit sourd type essorage lav l immeuble cause ventilation installer dernier étage couchage...,4.76
1,222887,beaucoup accueil superbe amie beaucoup aimer décoration vue magnifique très bien situer beaucoup,4.76
2,222887,très bel bien situer face porte bourgogne grand pièce vivre bel petit terrasse parfait reposer long promenade pied vélo gironde marché capucin moi...,4.76
3,222887,bel découverte HOTE très réactif,4.76
4,222887,passer très bon beaucoup charme très bien équiper terrasse exceptionnel très bien situer très proche tram commerce restaurant proximité attention ...,4.76


Si on gardait un avis par ligne, les avis d'un même logement se retrouveraient à la fois dans l'entraînement
et dans le test : le modèle pourrait reconnaître le logement au lieu d'apprendre ce qui fait un logement propre.

On met donc **bout à bout les avis de chaque logement** : 1 ligne = 1 logement = 1 note, comme 1 tweet = 1 étiquette au TD5.
On garde les logements avec **au moins 10 avis** : avec moins, le document est trop court pour juger un aspect.

## 3. La table finale

In [4]:
MIN_AVIS = 10

logements = ag.agreger_par_logement(df, MIN_AVIS)
print(len(logements), 'logements')
logements[['nb_avis', 'text_neg', 'quartier', 'review_scores_cleanliness',
           'review_scores_location', 'review_scores_value']].head()

5497 logements


,nb_avis,text_neg,quartier,review_scores_cleanliness,review_scores_location,review_scores_value
listing_id,,,,,,
222887,16,tres bien situer centre louer mais nuit très difficile bruit sourd type essorage lav l immeuble cause ventilation installer dernier étage couchage...,Bordeaux Sud,4.76,4.75,4.72
317273,12,connaître bien meilleur connaître plein centre proche magnifique parc public transport commun HOTE collègue disponible réactif efficace flexible l...,Chartrons - Grand Parc - Jardin Public,4.77,4.96,4.74
317658,11,HOTE répondre rapidement attente HOTE accueillir vraiment très serviable point négatif wifi NEG_fonctionner bien reviendron séjourner jour ami pro...,Centre ville (Bordeaux),4.84,4.97,4.77
333031,410,bon quartier studio parfait très bon accueil très bien impeccable bon accueil manque sec cheveu parfait très bien situer absolutely fin place to g...,Centre ville (Bordeaux),4.95,4.98,4.84
365993,49,venir colocataire semaine très bien passer propriétaire chaleureux serviable design spacieux fonctionnel terrasse agréable recommander très fortem...,Bgles,4.99,4.69,4.76


In [5]:
logements['nb_avis'].describe().round(1)

count    5497.0
mean       58.3
std        82.7
min        10.0
25%        17.0
50%        32.0
75%        66.0
max      2383.0
Name: nb_avis, dtype: float64

Exemple des avis du premier logement, mis bout à bout (version `text_neg`).

In [6]:
# Affiche les 600 premiers caractères du premier logement
print(logements['text_neg'].iloc[0][:600])

tres bien situer centre louer mais nuit très difficile bruit sourd type essorage lav l immeuble cause ventilation installer dernier étage couchage chambre entrée très très difficile contacter proprietaire trouver juste coté passer derniere nuit bien meilleur condition initiative malgré forcément décevoir gacher beaucoup accueil superbe amie beaucoup aimer décoration vue magnifique très bien situer beaucoup très bel bien situer face porte bourgogne grand pièce vivre bel petit terrasse parfait reposer long promenade pied vélo gironde marché capucin moins min pied vraiment plaisir aller faire cou


## 4. Représentation TF-IDF

TF-IDF donne à chaque mot un poids qui augmente avec sa fréquence dans le document et diminue
s'il apparaît dans beaucoup de documents. Paramètres (comme au TD5) :

- `strip_accents='unicode'` : "très" et "tres" deviennent le même mot. Les accents sont retirés
  ici et pas au nettoyage, car spaCy en avait besoin pour lemmatiser ;
- `min_df=5` : un mot présent dans moins de 5 logements (faute de frappe, nom rare) est ignoré ;
- unigrammes, puis unigrammes + bigrammes ("très propre", "neg_propre").

Ici, on décrit seulement la représentation. Pour les modèles (notebook 05), le TF-IDF sera appris
sur les données d'entraînement uniquement.

Taille du vocabulaire pour chaque version du texte :

In [7]:
lignes = []
for version in ag.VERSIONS:
    uni = TfidfVectorizer(strip_accents='unicode', min_df=5).fit(logements[version])
    bi = TfidfVectorizer(strip_accents='unicode', min_df=5, ngram_range=(1, 2)).fit(logements[version])
    lignes.append({'version': version, 'unigrammes': len(uni.vocabulary_), 'uni + bigrammes': len(bi.vocabulary_)})

pd.DataFrame(lignes).set_index('version')

,unigrammes,uni + bigrammes
version,,
comments_clean,12462,122571
text_lemmes,7957,89311
text_stop,7916,91438
text_neg,8744,90366
text_pos,8489,88598


Aperçu de la matrice TF-IDF (version `text_neg`) : une ligne par logement, une colonne par terme.

In [8]:
vect = TfidfVectorizer(strip_accents='unicode', min_df=5, ngram_range=(1, 2))
X = vect.fit_transform(logements['text_neg'])
termes = vect.get_feature_names_out()
print('matrice :', X.shape[0], 'logements x', X.shape[1], 'termes')

exemples = [t for t in ['propre', 'sale', 'neg_propre', 'calme', 'bruit'] if t in termes]
pd.DataFrame(X[:5].toarray(), index=logements.index[:5], columns=termes)[exemples].round(3)

matrice : 5497 logements x 90366 termes


,propre,sale,neg_propre,calme,bruit
listing_id,,,,,
222887,0.000,0.0,0.0,0.000,0.026
317273,0.000,0.0,0.0,0.033,0.000
317658,0.079,0.0,0.0,0.032,0.000
333031,0.134,0.0,0.0,0.060,0.000
365993,0.097,0.0,0.0,0.029,0.000


Les termes au poids le plus fort pour le logement le moins bien noté en propreté :

In [9]:
pire = logements.index.get_loc(logements['review_scores_cleanliness'].idxmin())
pd.Series(X[pire].toarray().ravel(), index=termes).nlargest(15).round(3)

court nuit            0.149
locataire             0.143
description contre    0.125
moins bruit           0.125
chambre               0.124
lit conforme          0.123
contre salle          0.121
peu                   0.120
faire pouvoir         0.119
plutot decevoir       0.119
savoir arriver        0.119
telephone passer      0.119
colloc                0.117
defaire lit           0.117
dormir faire          0.117
dtype: float64

Les termes avec négation les plus répandus (nombre de logements où ils apparaissent) :

In [10]:
presence = pd.Series((X > 0).sum(axis=0).A1, index=termes)
presence[presence.index.str.startswith('neg_')].nlargest(15)

neg_hesiter            3428
neg_redire             2178
neg_hesitation         2171
neg_probleme           1495
neg_loin               1449
neg_tres               1338
neg_manquer            1144
neg_souci              1052
neg_manque             1046
neg_hesiter revenir     999
neg_hesiter tres        802
neg_trop                682
neg_bruit               646
neg_pouvoir             575
neg_decevoir            573
dtype: int64

## 5. Sauvegarde

In [11]:
logements.to_parquet('../data/processed/logements.parquet')
print(logements.shape)

(5497, 15)
